# CVaR (worst-case loss) portfolio: a linear programming example, solved and checked

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProvenDecisions/optimization-examples/blob/main/notebooks/cvar-portfolio.ipynb)

Solved and checked by [ProvenDecisions](https://proven-decisions.ai/examples/cvar-portfolio?ref=github): the step-by-step walkthrough, the dual or KKT certificate and a version you can change and re-solve are on the example's page.

## The problem

Two funds had monthly returns (%) of A: 2, -3, 1, 4, -1 and B: -1, 2, 0, -2, 3 over five months. Invest fully, no shorts. Minimize the average loss over the worst two months of the resulting portfolio (loss = minus return).

## The model

Each line has a plain-language note.

$$
\begin{aligned}\min\;& t + 0.5 u_{1} + 0.5 u_{2} + 0.5 u_{3} + 0.5 u_{4} + 0.5 u_{5} &&\text{make this as small as possible}\\ \text{s.t.}\;& a + b = 1 &&\text{limit 1: left side must equal 1}\\ \;& u_{1} \ge - 2 a + b - t &&\text{limit 2: left side must reach at least the right side}\\ \;& u_{2} \ge 3 a - 2 b - t &&\text{limit 3: left side must reach at least the right side}\\ \;& u_{3} \ge - a - t &&\text{limit 4: left side must reach at least the right side}\\ \;& u_{4} \ge - 4 a + 2 b - t &&\text{limit 5: left side must reach at least the right side}\\ \;& u_{5} \ge a - 3 b - t &&\text{limit 6: left side must reach at least the right side}\\ \;& a \ge 0 &&\text{fraction of the portfolio invested in fund A cannot be negative}\\ \;& b \ge 0 &&\text{fraction of the portfolio invested in fund B cannot be negative}\\ \;& u_{1} \ge 0 &&\text{excess of month 1 loss over t (percent) cannot be negative}\\ \;& u_{2} \ge 0 &&\text{excess of month 2 loss over t (percent) cannot be negative}\\ \;& u_{3} \ge 0 &&\text{excess of month 3 loss over t (percent) cannot be negative}\\ \;& u_{4} \ge 0 &&\text{excess of month 4 loss over t (percent) cannot be negative}\\ \;& u_{5} \ge 0 &&\text{excess of month 5 loss over t (percent) cannot be negative}\end{aligned}
$$

## Solve it with cvxpy

Run the cell (in Colab, `cvxpy` is installed by the first line).

In [ ]:
%pip install -q cvxpy highspy

In [1]:
import cvxpy as cp

a = cp.Variable(name='a')  # fraction of the portfolio invested in fund A
b = cp.Variable(name='b')  # fraction of the portfolio invested in fund B
u1 = cp.Variable(name='u1')  # excess of month 1 loss over t (percent)
t = cp.Variable(name='t')  # value-at-risk threshold, the loss level in the tail formula (percent, free sign)
u2 = cp.Variable(name='u2')  # excess of month 2 loss over t (percent)
u3 = cp.Variable(name='u3')  # excess of month 3 loss over t (percent)
u4 = cp.Variable(name='u4')  # excess of month 4 loss over t (percent)
u5 = cp.Variable(name='u5')  # excess of month 5 loss over t (percent)

objective = cp.Minimize(t + 0.5*u1 + 0.5*u2 + 0.5*u3 + 0.5*u4 + 0.5*u5)
constraints = [
    a + b == 1,
    u1 >= -2*a + b - t,
    u2 >= 3*a - 2*b - t,
    u3 >= -a - t,
    u4 >= -4*a + 2*b - t,
    u5 >= a - 3*b - t,
    a >= 0,
    b >= 0,
    u1 >= 0,
    u2 >= 0,
    u3 >= 0,
    u4 >= 0,
    u5 >= 0,
]
problem = cp.Problem(objective, constraints)
problem.solve(solver='HIGHS')
print(problem.status, problem.value)
print('a =', a.value)
print('b =', b.value)
print('u1 =', u1.value)
print('t =', t.value)
print('u2 =', u2.value)
print('u3 =', u3.value)
print('u4 =', u4.value)
print('u5 =', u5.value)

optimal -0.13636363636363635
a = 0.36363636363636365
b = 0.6363636363636364
u1 = 0.09090909090909083
t = -0.18181818181818177
u2 = -0.0
u3 = -0.0
u4 = -0.0
u5 = -0.0


## The checked answer

The smallest objective value is **-0.136364**, at a = 0.363636, b = 0.636364, u1 = 0.0909091, t = -0.181818, u2 = 0, u3 = 0, u4 = 0, u5 = 0. All 13 constraints are satisfied (violation ≤0e+00). **Exactly proven optimal.**

**Problem class:** [LP](https://proven-decisions.ai/problem-classes?ref=github); **solver:** HIGHS. What makes the answer proven: a [dual or KKT certificate](https://proven-decisions.ai/glossary?ref=github), explained in plain words.

Step-by-step walkthrough, the proof of optimality and more: [CVaR (worst-case loss) portfolio: a linear programming example, solved and checked](https://proven-decisions.ai/examples/cvar-portfolio?ref=github)

**Change the numbers and solve it free:** [this model in the ProvenDecisions solver](https://proven-decisions.ai/?try=minimize%20t%20%2B%200.5%20u1%20%2B%200.5%20u2%20%2B%200.5%20u3%20%2B%200.5%20u4%20%2B%200.5%20u5%0Asubject%20to%0Aa%20%2B%20b%20%3D%201%0Au1%20%3E%3D%20-2%20a%20%2B%20b%20-%20t%0Au2%20%3E%3D%203%20a%20-%202%20b%20-%20t%0Au3%20%3E%3D%20-a%20-%20t%0Au4%20%3E%3D%20-4%20a%20%2B%202%20b%20-%20t%0Au5%20%3E%3D%20a%20-%203%20b%20-%20t%0Aa%2C%20b%2C%20u1%2C%20u2%2C%20u3%2C%20u4%2C%20u5%20%3E%3D%200%0A%23%20a%20%3D%20fraction%20of%20the%20portfolio%20invested%20in%20fund%20A%0A%23%20b%20%3D%20fraction%20of%20the%20portfolio%20invested%20in%20fund%20B%0A%23%20t%20%3D%20value-at-risk%20threshold%2C%20the%20loss%20level%20in%20the%20tail%20formula%20%28percent%2C%20free%20sign%29%0A%23%20u1%20%3D%20excess%20of%20month%201%20loss%20over%20t%20%28percent%29%0A%23%20u2%20%3D%20excess%20of%20month%202%20loss%20over%20t%20%28percent%29%0A%23%20u3%20%3D%20excess%20of%20month%203%20loss%20over%20t%20%28percent%29%0A%23%20u4%20%3D%20excess%20of%20month%204%20loss%20over%20t%20%28percent%29%0A%23%20u5%20%3D%20excess%20of%20month%205%20loss%20over%20t%20%28percent%29&ref=github)